# 04 - Join latent coordinates with metadata and inspect latent structure

This notebook prepares the learned latent representation for downstream analysis.

It performs six tasks:

1. Loads shared project configuration from `00_latent_space_config.ipynb`
2. Loads latent coordinates generated by the trained VAE
3. Loads aligned sample metadata from preprocessing
4. Verifies strict row-wise alignment between latent coordinates and metadata
5. Adds standardized comparison labels and plotting abbreviations
6. Writes a canonical working table and initial latent-space plots

This notebook is diagnostic rather than interpretive. Notebooks 5 and 6 should consume the same conventions established here.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)


In [ ]:
# Load shared latent-space notebook configuration.
# Keep 00_latent_space_config.ipynb in the same notebooks/ directory as this notebook.
%run ./00_latent_space_config.ipynb

print("Notebook directory       :", NOTEBOOK_DIR)
print("Project directory        :", PROJECT_DIR)
print("Processed data directory :", PROCESSED_DIR)
print("Notebook 04 plots dir    :", NB04_PLOTS_DIR)
print("Notebook 04 tables dir   :", NB04_TABLES_DIR)
print("Latent file exists?      :", LATENT_PATH.exists(), LATENT_PATH)
print("Metadata file exists?    :", METADATA_ALIGNED_PATH.exists(), METADATA_ALIGNED_PATH)


## Load latent coordinates and metadata


In [ ]:
df = build_latent_dataframe(
    latent_path=LATENT_PATH,
    metadata_path=METADATA_ALIGNED_PATH,
    require_full_contract=True,
)

df = add_standard_latent_labels(df)
latent_working = build_latent_working_table(df)

print("Joined latent/metadata shape :", df.shape)
print("Latent dimensionality        :", len(latent_columns(df)))
print("Working table shape          :", latent_working.shape)
display(latent_working.head())


## Alignment and metadata checks


In [ ]:
assert df.index.name == "sample_id", "Expected sample_id index."
assert df["sample_id"].astype(str).equals(df.index.to_series().astype(str)),     "sample_id column does not match dataframe index."

print("Alignment checks passed.")
print("Number of samples        :", df.shape[0])
print("Number of disease labels :", df["disease_clean"].nunique())
print("Number of tissue labels  :", df["tissue_clean"].nunique())

print("\nCondition counts:")
display(df["state"].value_counts(dropna=False))


## Validate shared comparison labels


In [ ]:
report_unmapped_latent_labels(df)

df_cancer = df[df["state"].eq(STATE_CANCER)].copy()

print("Cancer-only samples       :", df_cancer.shape[0])
print("Unique cancer diseases    :", df_cancer["disease"].nunique())
print("Unique project labels     :", df_cancer["short_label"].nunique())
print("Unique plot labels        :", df_cancer["plot_label"].nunique())

display(
    df_cancer[["tissue", "disease", "short_label", "plot_label"]]
    .drop_duplicates()
    .sort_values(["short_label", "plot_label"])
)


## Save canonical Notebook 4 working table


In [ ]:
working_table_path = NB04_TABLES_DIR / "latent_working_table.csv"
latent_working.to_csv(working_table_path, index=True)

print("Saved:", working_table_path)


## Static latent-space centroid plot


In [ ]:
plot_df = df_cancer.dropna(subset=["plot_label"]).copy()

plt.figure(figsize=(8, 6))

plt.scatter(
    plot_df["z1"],
    plot_df["z2"],
    s=15,
    alpha=0.3,
)

centroids = plot_df.groupby("plot_label")[["z1", "z2"]].mean()

plt.scatter(
    centroids["z1"],
    centroids["z2"],
    s=80,
)

for label, row in centroids.iterrows():
    plt.text(row["z1"], row["z2"], label, fontsize=9)

plt.xlabel("z1")
plt.ylabel("z2")
plt.title("Latent space: cancer samples with class centroids")
plt.tight_layout()

static_plot_path = NB04_PLOTS_DIR / "latent_space_centroids.png"
plt.savefig(static_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", static_plot_path)


## Interactive latent-space centroid plot


In [ ]:
try:
    import plotly.express as px
    import plotly.graph_objects as go

    fig = px.scatter(
        plot_df,
        x="z1",
        y="z2",
        color="plot_label",
        hover_data=[
            "short_label",
            "tissue",
            "disease",
            "comparison",
            "geo_accession",
            "title",
        ],
        title="Latent space with class centroids",
    )

    fig.update_traces(marker=dict(size=6, opacity=0.7))

    centroid_df = centroids.reset_index()
    fig.add_trace(
        go.Scatter(
            x=centroid_df["z1"],
            y=centroid_df["z2"],
            mode="markers+text",
            text=centroid_df["plot_label"],
            textposition="top center",
            marker=dict(size=8, symbol="x", line=dict(width=2)),
            name="Centroids",
            showlegend=False,
        )
    )

    fig.update_layout(
        height=650,
        legend=dict(
            x=0.99,
            y=0.99,
            xanchor="right",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.7)",
        ),
    )

    interactive_plot_path = NB04_PLOTS_DIR / "latent_space_interactive.html"
    fig.write_html(interactive_plot_path)
    fig.show()
    print("Saved:", interactive_plot_path)

except ImportError as exc:
    print("Plotly is not installed in this environment. Skipping interactive plot.")
    print(exc)


## Objects handed forward

Downstream notebooks should use:

- `df`: full latent + metadata table with standardized labels
- `df_cancer`: cancer-only subset
- `latent_working`: compact canonical table for Notebooks 5 and 6
- `latent_columns(df)`: shared helper for identifying latent dimensions
